# Lab 8.12 &mdash; Capstone: The Multi-Agent Chatbot

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Chain route -> real create_agent specialists -> synthesise into one handler
- Run a VOTE when two specialists conflict; gate any refund on a human
- Run the chatbot over a SUITE (general fallback + a conflict case) and score it

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; routing, synthesis, tool wiring, agent structure &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; The multi-agent customer-service chatbot](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-08-12"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Capstone: the **multi-agent customer-service chatbot** (the client's Lab 4.2), end to end. A
**supervisor** routes each message to the matching **specialists** (the **real `create_agent`** agents
from Lab 8.11); each returns a grounded finding; a **synthesiser** composes one reply. When two
specialists **conflict** on a sensitive call (a disputed refund), you don't paste the contradiction
&mdash; you run the **vote** from Lab 8.7 and **escalate a split** to a human. And anything
**irreversible** (a refund) is flagged **`needs_approval`** &mdash; never auto-done. You run it over a
**suite** (including a general fallback and a conflict) and score the outcomes. The routing, vote,
synthesis and refund gate are deterministic (graded); the real specialists run in the optional cell.

In [ ]:
from langchain_core.tools import tool

# The chatbot's context sources: invoices (order 4471 has a DUPLICATE charge) and known issues.
INVOICES = {
    "4471": [{"amount": 50, "date": "Jul 01"}, {"amount": 50, "date": "Jul 01"}],
    "5090": [{"amount": 30, "date": "Jul 02"}],
}
KNOWN_ISSUES = {
    "crash": {"bug": "BUG-231", "fix": "update to v4.2"},
    "login": {"bug": "BUG-118", "fix": "reset your password"},
}

from langchain_core.tools import tool
from collections import Counter

@tool
def lookup_invoice(order_id: str) -> list:
    """Look up the charges on an order by id."""
    return INVOICES.get(order_id, [])
@tool
def known_issues(symptom: str) -> dict:
    """Look up a known technical issue by symptom keyword."""
    for k, v in KNOWN_ISSUES.items():
        if k in symptom.lower():
            return v
    return {}

def route(message):
    m = message.lower()
    kws = {"billing": ("charg", "refund", "invoice", "billed"), "tech": ("crash", "bug", "login", "broken")}
    hits = [name for name, ks in kws.items() if any(k in m for k in ks)]
    return hits if hits else ["general"]

# The grounded findings each specialist produces (the offline stand-in for the REAL agents built below).
def billing_finding(message):
    return "order 5090 has one valid charge -> no refund" if "5090" in message.lower() else "duplicate charge on 4471 -> refund warranted"
def tech_finding(message):    return "matches BUG-231 -> update to v4.2"
def general_finding(message): return "forwarded to a human agent"
FINDINGS = {"billing": billing_finding, "tech": tech_finding, "general": general_finding}

def synthesize(findings):
    return " ".join(findings[k] for k in sorted(findings)) if findings else "No findings."

# Decision skills from earlier in the module, wired into the product here:
def is_dispute(message):   return "dispute" in message.lower()
def review_panel(message): return ["refund", "deny"]   # a disputed charge -> reviewers SPLIT (no clear majority)
def decide(verdicts, threshold=0.5):                    # Lab 8.7's vote
    top, n = Counter(verdicts).most_common(1)[0]
    if n / len(verdicts) > threshold: return {"decision": top, "escalate": False}
    return {"decision": None, "escalate": True}
print("tools, route, findings, synthesise & the vote are ready")

## Your Turn
Build the real `create_agent` specialists, then assemble `process` (route &rarr; run each specialist
&rarr; run the vote on a conflict &rarr; synthesise &rarr; gate the refund) and `evaluate`.

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

def build_specialist(tools, role):
    model = ChatOllama(model="llama3.2:1b")
    return create_agent(model, ___, system_prompt=f"You are the {role} specialist. Use only your tools.")   # TODO: bind this role own tools

billing_agent = build_specialist([lookup_invoice], 'billing')
tech_agent    = build_specialist([known_issues], 'tech')

def process(message):
    involved = route(message)
    findings = ___   # TODO: a dict mapping each involved specialist name to its finding for this message
    conflict = None
    if 'billing' in involved and is_dispute(message):
        verdicts = review_panel(message)
        conflict = ___   # TODO: run the vote over the verdicts (reuse decide from Lab 8.7)
        findings["billing"] = "reviewers split -> escalate to a human" if conflict["escalate"] else f"reviewers agree: {conflict['decision']}"
    reply = synthesize(findings)
    # a human decides if any finding warrants a refund OR the conflict vote escalated
    needs_human = ___   # TODO: True if any finding mentions "refund warranted" or "escalate"
    status = "needs_approval" if needs_human else "auto_ok"
    return {"agents": sorted(findings), "reply": reply, "status": status, "conflict": bool(conflict)}

SUITE = [
    {"message": "charged twice for 4471 and the app keeps crashing", "agents": ["billing", "tech"], "status": "needs_approval"},
    {"message": "the app keeps crashing on login",                  "agents": ["tech"],            "status": "auto_ok"},
    {"message": "please refund my invoice",                         "agents": ["billing"],         "status": "needs_approval"},
    {"message": "what are your hours?",                             "agents": ["general"],         "status": "auto_ok"},
    {"message": "why was I billed 30 on order 5090?",               "agents": ["billing"],         "status": "auto_ok"},
    {"message": "I want to dispute the charge on 4471",             "agents": ["billing"],         "status": "needs_approval"},
]

def evaluate():
    solved = ___   # TODO: count SUITE items where BOTH agents and status match the expected
    return solved, len(SUITE)

try:
    for t in SUITE:
        r = process(t['message'])
        print(r['agents'], '|', r['status'], '| conflict:', r['conflict'], '->', r['reply'][:44])
    print('score:', evaluate())
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("each specialist is a real create_agent (CompiledStateGraph)", lambda: type(billing_agent).__name__ == "CompiledStateGraph" and type(tech_agent).__name__ == "CompiledStateGraph")
expect_true("a two-intent message engages both specialists", lambda: process("charged twice and the app keeps crashing")["agents"] == ["billing", "tech"])
expect_true("a pure tech message engages only tech", lambda: process("the app keeps crashing")["agents"] == ["tech"])
expect_true("an unknown query falls back to the general agent", lambda: process("what are your hours?")["agents"] == ["general"])
expect_true("the reply is synthesised from the findings", lambda: (lambda r: "refund" in r.lower() and "bug-231" in r.lower())(process("charged twice and the app keeps crashing")["reply"]))
expect_true("a refund is gated on human approval", lambda: process("please refund my invoice")["status"] == "needs_approval")
expect_true("a valid (non-duplicate) charge is auto_ok", lambda: process("why was I billed 30 on order 5090?")["status"] == "auto_ok")
expect_true("a CONFLICT (dispute) runs a vote and escalates to a human", lambda: process("I want to dispute the charge on 4471")["conflict"] is True and process("I want to dispute the charge on 4471")["status"] == "needs_approval")
expect_true("the chatbot solves the whole suite", lambda: evaluate() == (len(SUITE), len(SUITE)))

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Run the REAL create_agent specialists behind the supervisor: route, invoke each matching agent, then apply the conflict vote and refund gate you built. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
def specialist_reply(agent, message):
    result = agent.invoke({"messages": [{"role": "user", "content": message}]}, config={"recursion_limit": 8})
    return result["messages"][-1].content
try:
    if ollama_up():
        msg = "I was charged twice for order 4471 and the app keeps crashing."
        agents = {"billing": billing_agent, "tech": tech_agent}
        live_findings = {name: specialist_reply(agents[name], msg) for name in route(msg) if name in agents}
        print("live findings:", live_findings)
        print("(offline, process() already scored the whole suite -- routing, the conflict vote, synthesis and the refund gate.)")
    else:
        print("No Ollama reachable -- skipping the live run. The offline chatbot above already ran the whole suite:")
        print("routed every ticket, voted on the conflict case, synthesised one reply, and gated every refund on a human.")
    print("Next: Day 5 -- agents in industry (finance / health / cyber) and responsible AI.")
except Exception as e:
    print("Live run skipped:", type(e).__name__)

---
### Key takeaway
You built a multi-agent customer-service chatbot end to end -- specialists coordinated by a supervisor, a vote when they conflict, findings synthesised into one reply, refunds gated on a human. That completes Day 4. Next: Day 5 -- agents in the real world, responsibly.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>